# 00. Project Setup & Data Download

This notebook configures the local working environment and downloads the raw AAPL dataset using `yfinance`.

In [1]:
!pip install yfinance xgboost scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\yeapz\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Local Environment Setup

We structure the project using local paths following the implementation plan.

In [2]:
# ── LOCAL DEV SETUP ──────────────────────────────────────────────────────────
import os

PROJECT_ROOT = os.path.abspath('.')
DATA_DIR     = os.path.join(PROJECT_ROOT, 'data')
FIGURES_DIR  = os.path.join(PROJECT_ROOT, 'outputs', 'figures')
METRICS_DIR  = os.path.join(PROJECT_ROOT, 'outputs', 'metrics')
PREDS_DIR    = os.path.join(PROJECT_ROOT, 'outputs', 'predictions')

for d in [DATA_DIR, FIGURES_DIR, METRICS_DIR, PREDS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── PROJECT CONSTANTS ─────────────────────────────────────────────────────────
TICKER       = 'AAPL'
# Flaw Fix: We pad the start date slightly into 2016 to absorb the 26-day MACD burn-in.
# This ensures that our actual clean dataset begins precisely on 2017-01-01.
FETCH_START  = '2016-11-15' 
START_DATE   = '2017-01-01'
END_DATE     = '2024-12-31'
TRAIN_FRAC   = 0.80
RANDOM_STATE = 42

print(f"Data Directory configured at: {DATA_DIR}")

Data Directory configured at: C:\Users\yeapz\OneDrive\Documents\data mining assignment\data


## 2. Download Raw Data

We download the daily prices. Modern `yfinance` returns a MultiIndex format which we will flatten for compatibility.

In [3]:
import yfinance as yf
import pandas as pd

print(f"Downloading {TICKER} from {FETCH_START} to {END_DATE}...")
df_raw = yf.download(TICKER, start=FETCH_START, end=END_DATE)

# Flaw Fix: yfinance recently changed output defaults. It now returns a MultiIndex.
# We flatten the columns to match expected standard OHLCV.
if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = df_raw.columns.get_level_values(0)

# Ensure only necessary columns are kept
expected_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
df_raw = df_raw[expected_cols]

# Reset index so 'Date' becomes a column
df_raw.reset_index(inplace=True)

df_raw.head()

C:\Users\yeapz\AppData\Local\Temp\ipykernel_20488\1017127365.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_raw = yf.download(TICKER, start=FETCH_START, end=END_DATE)


[*********************100%***********************]  1 of 1 completed

Price,Date,Open,High,Low,Close,Volume
0,2016-11-15,24.539873,24.795473,24.445463,24.664219,129058000
1,2016-11-16,24.569799,25.382653,24.546772,25.327387,235362000
2,2016-11-17,25.285944,25.410290,25.060281,25.318182,110528000
3,2016-11-18,25.265219,25.454040,25.251403,25.343510,113715600
4,2016-11-21,25.357327,25.787931,25.331997,25.728062,117058400


## 3. Data Diagnostics

In [4]:
print("Shape of raw data:", df_raw.shape)
print("\nData Types:\n", df_raw.dtypes)
print("\nMissing Values:\n", df_raw.isnull().sum())

Shape of raw data: (2043, 6)

Data Types:
 Price
Date      datetime64[ns]
Open             float64
High             float64
Low              float64
Close            float64
Volume             int64
dtype: object

Missing Values:
 Price
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64


## 4. Save to Disk

In [5]:
# Save the raw dataset
raw_data_path = os.path.join(DATA_DIR, 'raw_stock_data.csv')
df_raw.to_csv(raw_data_path, index=False)
print(f"✅ Successfully saved to {raw_data_path}")

✅ Successfully saved to C:\Users\yeapz\OneDrive\Documents\data mining assignment\data\raw_stock_data.csv
